# Underwater enhancement: FUnIE-GAN

Runs an underwater `.mp4` through **FUnIE-GAN** (Fast Underwater Image Enhancement GAN) and shows a before and after comparison of the frames.

**What FUnIE-GAN is:** a conditional GAN (generative adversarial network), published as *Fast Underwater Image Enhancement for Improved Visual Perception* (Islam, Xia, Sattar, IEEE RA-L 2020). It corrects the colour cast, contrast loss and haze typical of underwater footage.

**No training is needed.** The [official repository](https://github.com/xahidbuffon/FUnIE-GAN) is released under the **MIT licence** and ships pretrained generator weights (`funie_generator.pth`) trained on the EUVP dataset. This notebook downloads those official weights and uses them for inference only. The generator architecture (a 5-layer U-Net) is reproduced in `utils/funie_utils.py` and must match the checkpoint's layer shapes exactly for `load_state_dict` to work.

Workflow:
- Load the pretrained generator
- Extract frames from the video, enhance them and reassemble an enhanced video
- Compare original and enhanced frames side by side
- Test three ways of handling the 256x256 network size: raw output, linear versus cubic upscaling, and inference near native resolution

The controlled comparison against expert annotations is in `zeroshot/coral_enhancement_comparison.ipynb`.

In [ ]:
import os
import sys
from glob import glob

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

sys.path.append("../utils")   # shared modules live in the repository's utils folder
from funie_utils import load_funie_generator, INFER_SIZE

# Apple GPU (MPS) when available, otherwise CPU
DEVICE = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print("Using", "Mac GPU (MPS)" if DEVICE.type == "mps" else "CPU")

## Load the pretrained generator

`load_funie_generator` downloads `funie_generator.pth` (about 28 MB) from the official `xahidbuffon/FUnIE-GAN` repository on first use, then builds the generator and loads the weights.

In [ ]:
OUTPUT_ROOT = "outputs/funie"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
WEIGHTS_PATH = os.path.join(OUTPUT_ROOT, "funie_generator.pth")

generator = load_funie_generator(WEIGHTS_PATH, DEVICE)
print(f"Loaded pretrained FUnIE-GAN generator ({os.path.getsize(WEIGHTS_PATH) / 1e6:.1f} MB of weights)")

## Pre- and post-processing

The model was trained on 256x256 crops normalised to [-1, 1]. Each frame is resized to 256x256 for inference (matching the paper's setup), then the enhanced output is resized back up to the original resolution.

In [ ]:
def preprocess(frame_bgr):
    """OpenCV BGR frame -> normalised tensor ready for the generator."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, INFER_SIZE, interpolation=cv2.INTER_AREA)
    arr = resized.astype(np.float32) / 127.5 - 1.0          # [0,255] -> [-1,1]
    tensor = torch.from_numpy(arr.transpose(2, 0, 1)).unsqueeze(0)  # (1,3,H,W)
    return tensor.to(DEVICE)

def deprocess(tensor, out_size=None):
    """Generator output tensor -> uint8 BGR frame (OpenCV-ready)."""
    arr = tensor.squeeze(0).detach().cpu().numpy().transpose(1, 2, 0)  # (H,W,3)
    arr = ((arr + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    if out_size is not None:
        arr = cv2.resize(arr, out_size, interpolation=cv2.INTER_CUBIC)
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)

@torch.no_grad()
def enhance_frame(frame_bgr, out_size=None):
    inp = preprocess(frame_bgr)
    gen = generator(inp)
    return deprocess(gen, out_size=out_size)

## Choose the video

`INPUT_VIDEO` points at the `.mp4` to enhance. `FRAME_STRIDE` skips frames (for example `2` processes every other frame), since GAN inference is the slow part, especially on CPU. `MAX_FRAMES` caps how many frames are processed (`None` processes the whole video).

In [ ]:
INPUT_VIDEO = "../data/bluerov_videos/Shortened.mp4"
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, "resize_256")
FRAME_STRIDE = 1                    # process every Nth frame (1 = every frame)
MAX_FRAMES = None                   # cap total frames processed, or None for no cap

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "original_frames"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "enhanced_frames"), exist_ok=True)

assert os.path.exists(INPUT_VIDEO), f"Video not found at {INPUT_VIDEO}. Update INPUT_VIDEO above."

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Video: {width}x{height} @ {fps:.2f} fps, {total_frames} frames")

## Extract frames, enhance them and reassemble the video

Reads the video frame by frame, runs the frames selected by `FRAME_STRIDE` and `MAX_FRAMES` through the generator, and writes both an enhanced video and the individual before and after frame images.

In [ ]:
out_video_path = os.path.join(OUTPUT_DIR, "enhanced_video.mp4")
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
effective_fps = fps / FRAME_STRIDE
writer = cv2.VideoWriter(out_video_path, fourcc, effective_fps, (width, height))

frame_idx = 0
saved_idx = 0
pbar = tqdm(total=(MAX_FRAMES or total_frames))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % FRAME_STRIDE == 0:
        enhanced = enhance_frame(frame, out_size=(width, height))
        writer.write(enhanced)

        cv2.imwrite(os.path.join(OUTPUT_DIR, "original_frames", f"frame_{saved_idx:05d}.png"), frame)
        cv2.imwrite(os.path.join(OUTPUT_DIR, "enhanced_frames", f"frame_{saved_idx:05d}.png"), enhanced)

        saved_idx += 1
        pbar.update(1)
        if MAX_FRAMES is not None and saved_idx >= MAX_FRAMES:
            break
    frame_idx += 1

pbar.close()
cap.release()
writer.release()
print(f"\nProcessed {saved_idx} frames.")
print(f"Enhanced video saved to: {out_video_path}")

## Visual before and after comparison

Samples a handful of frames spread evenly across the processed clip and shows original versus enhanced side by side.

In [ ]:
orig_paths = sorted(glob(os.path.join(OUTPUT_DIR, "original_frames", "*.png")))
enh_paths = sorted(glob(os.path.join(OUTPUT_DIR, "enhanced_frames", "*.png")))

N_SAMPLES = min(5, len(orig_paths))
sample_idxs = np.linspace(0, len(orig_paths) - 1, N_SAMPLES).astype(int)

fig, axes = plt.subplots(N_SAMPLES, 2, figsize=(8, 3.2 * N_SAMPLES))
if N_SAMPLES == 1:
    axes = axes[None, :]

for row, i in enumerate(sample_idxs):
    orig = cv2.cvtColor(cv2.imread(orig_paths[i]), cv2.COLOR_BGR2RGB)
    enh = cv2.cvtColor(cv2.imread(enh_paths[i]), cv2.COLOR_BGR2RGB)

    axes[row, 0].imshow(orig)
    axes[row, 0].set_title(f"Original — frame {i}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(enh)
    axes[row, 1].set_title(f"FUnIE-GAN enhanced — frame {i}")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

## Resampling and resolution tests

The network works on 256x256 input, so a 16:9 frame is squashed to a square and stretched back, which smooths fine texture. Three variations were compared (they correspond to the FUnIE-GAN enhancement figure in the report):
- raw 256x256 model output, without upscaling
- the same output upscaled with linear instead of cubic interpolation
- inference at near-native resolution (rounded down to a multiple of 32), which needs only minimal upscaling

In [ ]:
# Raw 256x256 model output, without upscaling
@torch.no_grad()
def enhance_frame_raw(frame_bgr):
    inp = preprocess(frame_bgr) # resizes down to 256x256 internally
    gen = generator(inp)
    return deprocess(gen, out_size=None) # no out_size, stays 256x256

test_frame = cv2.imread(orig_paths[0])
raw_256 = enhance_frame_raw(test_frame)
plt.imshow(cv2.cvtColor(raw_256, cv2.COLOR_BGR2RGB))
plt.title("Raw 256x256 model output (no upscale)")
plt.axis("off")
plt.show()

In [ ]:
# Linear interpolation instead of cubic for the upscale
def deprocess_linear(tensor, out_size=None):
    arr = tensor.squeeze(0).detach().cpu().numpy().transpose(1, 2, 0)
    arr = ((arr + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    if out_size is not None:
        arr = cv2.resize(arr, out_size, interpolation=cv2.INTER_LINEAR)
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)

@torch.no_grad()
def enhance_frame_linear(frame_bgr, out_size=None):
    inp = preprocess(frame_bgr)
    gen = generator(inp)
    return deprocess_linear(gen, out_size=out_size)

test_linear = enhance_frame_linear(test_frame, out_size=(width, height))
plt.imshow(cv2.cvtColor(test_linear, cv2.COLOR_BGR2RGB))
plt.title("Upscaled with INTER_LINEAR instead of INTER_CUBIC")
plt.axis("off")
plt.show()

In [ ]:
# Inference at near-native resolution
def round_down_to_32(x):
    return (x // 32) * 32

native_w, native_h = round_down_to_32(width), round_down_to_32(height)  # e.g. 1920x1080 -> 1920x1056

def preprocess_native(frame_bgr, size):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, size, interpolation=cv2.INTER_AREA)  # size = (w, h), close to original
    arr = resized.astype(np.float32) / 127.5 - 1.0
    tensor = torch.from_numpy(arr.transpose(2, 0, 1)).unsqueeze(0)
    return tensor.to(DEVICE)

@torch.no_grad()
def enhance_frame_native(frame_bgr, size, out_size=None):
    inp = preprocess_native(frame_bgr, size)
    gen = generator(inp)
    return deprocess(gen, out_size=out_size)  # upscale the small gap from native_w/h back to width/height if needed

test_native = enhance_frame_native(test_frame, (native_w, native_h), out_size=(width, height))
plt.imshow(cv2.cvtColor(test_native, cv2.COLOR_BGR2RGB))
plt.title(f"Inference at native res ({native_w}x{native_h}), minimal upscale")
plt.axis("off")
plt.show()

## Full run at near-native resolution

In [ ]:
# Full run using the near-native-resolution approach (helper functions are defined in the previous section)

# Re-run the video processing
cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

native_w, native_h = round_down_to_32(width), round_down_to_32(height)
print(f"Native inference size: {native_w}x{native_h} (original: {width}x{height})")

OUTPUT_DIR_V2 = os.path.join(OUTPUT_ROOT, "native")
os.makedirs(OUTPUT_DIR_V2, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR_V2, "original_frames"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR_V2, "enhanced_frames"), exist_ok=True)

out_video_path_v2 = os.path.join(OUTPUT_DIR_V2, "enhanced_video_native.mp4")
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
effective_fps = fps / FRAME_STRIDE
writer = cv2.VideoWriter(out_video_path_v2, fourcc, effective_fps, (width, height))

frame_idx = 0
saved_idx = 0
pbar = tqdm(total=(MAX_FRAMES or total_frames))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % FRAME_STRIDE == 0:
        enhanced = enhance_frame_native(frame, (native_w, native_h), out_size=(width, height))
        writer.write(enhanced)

        cv2.imwrite(os.path.join(OUTPUT_DIR_V2, "original_frames", f"frame_{saved_idx:05d}.png"), frame)
        cv2.imwrite(os.path.join(OUTPUT_DIR_V2, "enhanced_frames", f"frame_{saved_idx:05d}.png"), enhanced)

        saved_idx += 1
        pbar.update(1)
        if MAX_FRAMES is not None and saved_idx >= MAX_FRAMES:
            break
    frame_idx += 1

pbar.close()
cap.release()
writer.release()
print(f"\nProcessed {saved_idx} frames.")
print(f"Enhanced video saved to: {out_video_path_v2}")